# 1 · Encode a light curve

EncoTESS turns a single TESS 2-minute light curve into a fixed-length **1536-dimensional
latent vector**. The encoder is a bidirectional MinGRU that reads the flux time series
(conditioned on the observation times and a handful of stellar metadata fields) and a
time-aware pooling step that collapses its per-timestep hidden states into one vector.

This notebook walks through:

1. preparing a light curve and its metadata,
2. encoding it to the 1536-d latent,
3. re-expressing that latent in the global **PCA basis** the package ships in.

The only inputs the encoder needs are three equal-length arrays — `flux`, `flux_err`,
`time` — plus an optional metadata dictionary.

In [ ]:
import numpy as np
import encotess

# A toy light curve stands in for real data here. In practice, pass your own
# arrays: normalized flux, its per-point uncertainty, and the timestamps in days.
rng = np.random.default_rng(0)
n = 6000
time = (np.arange(n) * (2.0 / 60.0 / 24.0)).astype(np.float32)   # 2-min cadence, in days
flux = 0.8 * np.sin(2 * np.pi * time / 3.2) + rng.normal(0, 1.0, n)
flux = ((flux - flux.mean()) / flux.std()).astype(np.float32)     # mean 0, unit scale
flux_err = np.ones(n, dtype=np.float32)

print('flux/flux_err/time lengths:', len(flux), len(flux_err), len(time))

### Metadata

The encoder was trained with a small set of stellar/instrumental fields alongside the
light curve. Provide them as a dictionary keyed by `encotess.DEFAULT_METADATA_FIELDS`.
You can omit any field you don't have (or set it to `NaN`) — the encoder was trained
with a mask channel, so it handles missing values gracefully.

In [ ]:
print('metadata fields:', encotess.DEFAULT_METADATA_FIELDS)

meta = {
    'cadence_s': 120.0, 'Tmag': 10.5, 'sector': 45, 'camera': 1, 'ccd': 2,
    'parallax': 5.0, 'parallax_error': 0.02, 'G0': 10.8, 'G0_err': 0.01,
    'BPRP0': 1.1, 'BPRP0_err': 0.03, 'median_flux': 1.0e4, 'iqr_half_flux': 50.0,
}

### Encode

`load_encoder()` loads the bundled trained weights (CPU by default; pass
`device='cuda'` if you have a GPU). `encode(...)` returns the 1536-d latent.

In [ ]:
enc = encotess.load_encoder(device='cpu')
z = enc.encode(flux, flux_err, time, metadata=meta)

print('latent shape:', z.shape)
print('latent L2 norm:', float((z ** 2).sum() ** 0.5))

### The PCA basis

The 1536-d latent has strongly correlated dimensions. The package ships a single
global **PCA** — fit once over every released light curve — that re-expresses the latent
in a decorrelated, variance-ordered basis. It is *unwhitened* and *full-rank*, so it is a
lossless orthonormal rotation of the (standardized) latent: no information is discarded,
the components are just reordered by how much variance they carry.

`project_pca(z, dim=k)` keeps the top-`k` components. The top few already capture most
of the variance.

In [ ]:
z16 = enc.project_pca(z, dim=16)     # compact
z64 = enc.project_pca(z, dim=64)     # the bundled 'preview' width
zfull = enc.project_pca(z)           # dim=None -> all 1536 components

print('pca-16:', z16.shape, ' pca-64:', z64.shape, ' full:', zfull.shape)
print('first 4 PCs:', z16[:4].round(3))

# Cumulative explained variance, read from the shipped PCA artifact.
from encotess import assets
pca_npz = np.load(assets.pca_weights_path(), allow_pickle=False)
cum = np.cumsum(pca_npz['pca_explained_variance_ratio'])
print(f'cumulative variance:  top-16 = {cum[15]:.1%},  top-64 = {cum[63]:.1%},  all = {cum[-1]:.1%}')

### Lossless round-trip

Because the full-rank PCA is an orthonormal rotation, we can invert it and recover the
original latent to floating-point precision — a quick sanity check that nothing is lost.

In [ ]:
gp = enc._pca                                        # the loaded GlobalPCA
x_norm = zfull.astype(np.float64) @ gp.components + gp.pca_mean
x_rec = x_norm * (gp.X_std + 1e-8) + gp.X_mean       # undo the per-feature standardization
print('max |reconstruction - original|:', float(np.abs(x_rec - z).max()))

### Next

- **[2 · Explore the released encodings](02_explore_the_encodings.ipynb)** — the package
  also ships the PCA encoding and a 2-D UMAP for *every* released light curve.
- **[3 · Forecast flux](03_predict_flux.ipynb)** — use the model's flow head to predict
  flux with an uncertainty band.